In [1]:
# ============================================================
# CELL 1 — SETUP AND LOAD SELECTED FEATURES
# Notebook: 05_model_training.ipynb
# ============================================================

import numpy as np
import pandas as pd
import os
import sys
import warnings
import time
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, classification_report,
    confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight


COMPONENT_PATH = r"C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals"
sys.path.insert(0, os.path.join(COMPONENT_PATH, "src"))
from utils.config import RESULTS_PATH

# ── Load selected feature matrix ─────────────────────────────
csv_path    = os.path.join(
    RESULTS_PATH, "objective1",
    "wesad_features_selected.csv"
)
features_df = pd.read_csv(csv_path)

feature_cols = [c for c in features_df.columns
                if c not in ['label', 'subject_id']]

X        = features_df[feature_cols].values
y        = features_df['label'].values
subjects = features_df['subject_id'].values

print("=" * 65)
print("MODEL TRAINING — SETUP")
print("=" * 65)
print(f"Feature matrix: {features_df.shape}")
print(f"Features used:  {len(feature_cols)}")
print(f"Feature names:  {feature_cols}")
print()
print("LABEL DISTRIBUTION:")
for lid, name in {
    0:'Normal', 1:'Pre-stress', 2:'Stress'
}.items():
    count = (y == lid).sum()
    pct   = count / len(y) * 100
    print(f"  Class {lid} ({name:10s}): "
          f"{count:4d} windows ({pct:.1f}%)")

print()
print("SUBJECTS:")
unique_subjects = sorted(np.unique(subjects))
print(f"  {unique_subjects}")
print(f"  Total: {len(unique_subjects)} subjects")

print()
print("CLASS WEIGHTS (for handling imbalance):")
classes   = np.array([0, 1, 2])
class_wts = compute_class_weight(
    'balanced', classes=classes, y=y
)
for i, (c, w) in enumerate(zip(classes, class_wts)):
    name = ['Normal', 'Pre-stress', 'Stress'][i]
    print(f"  Class {c} ({name:10s}): weight = {w:.3f}")

print()
print("Cell 1 complete — ready to define LOSO and models")

Config loaded successfully
MODEL TRAINING — SETUP
Feature matrix: (883, 24)
Features used:  22
Feature names:  ['AVNN', 'MeanHR', 'wavelet_d3_energy', 'SDHR', 'SCR_count', 'SCR_width_mean', 'EDA_q75', 'SCR_amp_mean', 'EDA_mean', 'SCL_mean', 'ACC_mean', 'ACC_std', 'EDA_slope', 'deriv1_std', 'deriv2_mean', 'pNN20', 'EDA_q10', 'EDA_q90', 'EDA_std', 'wavelet_d1_energy', 'EDA_range', 'SD1']

LABEL DISTRIBUTION:
  Class 0 (Normal    ):  587 windows (66.5%)
  Class 1 (Pre-stress):   29 windows (3.3%)
  Class 2 (Stress    ):  267 windows (30.2%)

SUBJECTS:
  [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]
  Total: 15 subjects

CLASS WEIGHTS (for handling imbalance):
  Class 0 (Normal    ): weight = 0.501
  Class 1 (Pre-stress): weight = 10.149
  Class 2 (Stress    ): weight = 1.102

Cell 1 complete — ready to define LOSO and models


In [2]:
# ============================================================
# CELL 2 — LOSO CROSS-VALIDATION FUNCTION
#
# What it does:
#   For each subject S in [S2...S17]:
#     Train on all other 14 subjects
#     Test on subject S
#     Record predictions and true labels
#   Return all predictions across all 15 rounds
#
# Why StandardScaler inside LOSO:
#   WRONG: scale all data then split
#     Test subject's stats leaked into scaler
#     Model indirectly saw test data
#     Inflated performance
#
#   RIGHT: scale only inside each fold
#     Fit scaler on 14 training subjects only
#     Apply same scaler to test subject
#     No information leakage
#     True held-out evaluation
#
# Example:
#   Fold 1 (test=S2):
#     Scaler fitted on S3-S17 data
#     S3-S17 AVNN range: 650-950ms → scale to 0-1
#     S2's AVNN 820ms mapped using S3-S17 scale
#     S2 never influenced the scaling
# ============================================================

def run_loso(model_class, model_params, X, y, subjects,
             feature_cols, model_name="Model"):
    """
    Run LOSO cross-validation for a given model.

    Parameters:
        model_class  : sklearn model class (e.g. RandomForestClassifier)
        model_params : dict of hyperparameters
        X            : feature matrix (n_windows × n_features)
        y            : labels (n_windows,)
        subjects     : subject ID per window (n_windows,)
        feature_cols : feature names
        model_name   : string for printing

    Returns:
        results dict with all metrics
    """

    unique_subjects = sorted(np.unique(subjects))
    all_true        = []
    all_pred        = []
    fold_f1_macro   = []
    fold_f1_pre     = []

    for sid in unique_subjects:

        # ── Split ────────────────────────────────────────────
        test_mask  = subjects == sid
        train_mask = ~test_mask

        X_train = X[train_mask]
        y_train = y[train_mask]
        X_test  = X[test_mask]
        y_test  = y[test_mask]

        # ── Scale INSIDE fold ─────────────────────────────────
        # Fit only on training data — no leakage
        scaler  = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

        # ── Train model ───────────────────────────────────────
        model = model_class(**model_params)
        model.fit(X_train, y_train)

        # ── Predict ───────────────────────────────────────────
        y_pred = model.predict(X_test)

        # ── Record ───────────────────────────────────────────
        all_true.extend(y_test.tolist())
        all_pred.extend(y_pred.tolist())

        # Fold-level metrics
        fold_f1 = f1_score(
            y_test, y_pred,
            average='macro',
            zero_division=0
        )
        fold_f1_pre = f1_score(
            y_test, y_pred,
            labels=[1],
            average='macro',
            zero_division=0
        )
        fold_f1_macro.append(fold_f1)

    # ── Overall metrics ───────────────────────────────────────
    all_true = np.array(all_true)
    all_pred = np.array(all_pred)

    f1_macro = f1_score(
        all_true, all_pred,
        average='macro',
        zero_division=0
    )
    f1_pre = f1_score(
        all_true, all_pred,
        labels=[1],
        average='macro',
        zero_division=0
    )
    f1_normal = f1_score(
        all_true, all_pred,
        labels=[0],
        average='macro',
        zero_division=0
    )
    f1_stress = f1_score(
        all_true, all_pred,
        labels=[2],
        average='macro',
        zero_division=0
    )

    # Ordinal MAE
    ordinal_mae = np.mean(np.abs(all_true - all_pred))

    # Accuracy
    accuracy = np.mean(all_true == all_pred)

    return {
        'model_name':  model_name,
        'f1_macro':    f1_macro,
        'f1_normal':   f1_normal,
        'f1_prestress': f1_pre,
        'f1_stress':   f1_stress,
        'ordinal_mae': ordinal_mae,
        'accuracy':    accuracy,
        'all_true':    all_true,
        'all_pred':    all_pred,
        'fold_f1':     fold_f1_macro
    }


print("LOSO function defined")
print()
print("What LOSO does per fold:")
print("  1. Separate one subject as test")
print("  2. Fit StandardScaler on 14 training subjects")
print("  3. Scale both train and test with same scaler")
print("  4. Train model on scaled training data")
print("  5. Predict on scaled test subject")
print("  6. Record predictions")
print("  7. Repeat for all 15 subjects")
print()
print("Cell 2 complete — LOSO ready")

LOSO function defined

What LOSO does per fold:
  1. Separate one subject as test
  2. Fit StandardScaler on 14 training subjects
  3. Scale both train and test with same scaler
  4. Train model on scaled training data
  5. Predict on scaled test subject
  6. Record predictions
  7. Repeat for all 15 subjects

Cell 2 complete — LOSO ready


In [3]:
# ============================================================
# CELL 3 — DEFINE ALL 10 MODELS
#
# Each model uses class_weight='balanced' where possible
# This compensates for 18.6:1 Normal:Pre-stress imbalance
#
# Models chosen based on literature:
#   Papers 10, 11: RF and XGBoost top performers
#   Paper 12:      SVM good for physiological data
#   Paper 14:      LSTM for sequential data
#   Paper 15:      Broad comparison across model types
# ============================================================

from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

# Compute class weights for XGBoost
n_normal    = (y == 0).sum()
n_prestress = (y == 1).sum()
n_stress    = (y == 2).sum()

# XGBoost uses scale_pos_weight differently for multiclass
# Use sample_weight during training instead
xgb_scale   = n_normal / n_prestress  # focus on minority class

# Define all 10 models with their parameters
models = [
    (
        "Random Forest",
        RandomForestClassifier,
        {
            'n_estimators':  300,
            'max_depth':     None,
            'class_weight':  'balanced',
            'random_state':  42,
            'n_jobs':        -1
        }
    ),
    (
        "XGBoost",
        xgb.XGBClassifier,
        {
            'n_estimators':    300,
            'max_depth':       6,
            'learning_rate':   0.1,
            'use_label_encoder': False,
            'eval_metric':     'mlogloss',
            'random_state':    42,
            'n_jobs':          -1,
            'verbosity':       0
        }
    ),
    (
        "SVM-RBF",
        SVC,
        {
            'kernel':       'rbf',
            'C':            10,
            'gamma':        'scale',
            'class_weight': 'balanced',
            'random_state': 42
        }
    ),
    (
        "SVM-Linear",
        SVC,
        {
            'kernel':       'linear',
            'C':            1.0,
            'class_weight': 'balanced',
            'random_state': 42
        }
    ),
    (
        "KNN",
        KNeighborsClassifier,
        {
            'n_neighbors': 7,
            'weights':     'distance',
            'n_jobs':      -1
        }
    ),
    (
        "LDA",
        LinearDiscriminantAnalysis,
        {
            'solver': 'svd'
        }
    ),
    (
        "AdaBoost",
        AdaBoostClassifier,
        {
            'n_estimators': 200,
            'learning_rate': 0.5,
            'random_state':  42
        }
    ),
    (
        "MLP",
        MLPClassifier,
        {
            'hidden_layer_sizes': (128, 64, 32),
            'activation':         'relu',
            'max_iter':           500,
            'random_state':       42,
            'early_stopping':     True,
            'validation_fraction': 0.1
        }
    ),
    (
        "Logistic Regression",
        LogisticRegression,
        {
            'C':            1.0,
            'class_weight': 'balanced',
            'max_iter':     1000,
            'random_state': 42,
            'n_jobs':       -1
        }
    ),
    (
        "Gradient Boosting",
        GradientBoostingClassifier,
        {
            'n_estimators':  200,
            'max_depth':     4,
            'learning_rate': 0.1,
            'random_state':  42
        }
    ),
]

print("=" * 65)
print("10 MODELS DEFINED")
print("=" * 65)
for i, (name, cls, params) in enumerate(models):
    print(f"  {i+1:2d}. {name}")
    print(f"      Key params: "
          f"{list(params.keys())[:4]}")
print()
print("Cell 3 complete — ready to run LOSO training")

10 MODELS DEFINED
   1. Random Forest
      Key params: ['n_estimators', 'max_depth', 'class_weight', 'random_state']
   2. XGBoost
      Key params: ['n_estimators', 'max_depth', 'learning_rate', 'use_label_encoder']
   3. SVM-RBF
      Key params: ['kernel', 'C', 'gamma', 'class_weight']
   4. SVM-Linear
      Key params: ['kernel', 'C', 'class_weight', 'random_state']
   5. KNN
      Key params: ['n_neighbors', 'weights', 'n_jobs']
   6. LDA
      Key params: ['solver']
   7. AdaBoost
      Key params: ['n_estimators', 'learning_rate', 'random_state']
   8. MLP
      Key params: ['hidden_layer_sizes', 'activation', 'max_iter', 'random_state']
   9. Logistic Regression
      Key params: ['C', 'class_weight', 'max_iter', 'random_state']
  10. Gradient Boosting
      Key params: ['n_estimators', 'max_depth', 'learning_rate', 'random_state']

Cell 3 complete — ready to run LOSO training


In [4]:
# ============================================================
# CELL 4 — RUN LOSO ON ALL 10 MODELS
#
# This is the main training cell
# Each model is trained 15 times (once per subject)
# Results stored in all_results list
# ============================================================

print("=" * 65)
print("RUNNING LOSO CROSS-VALIDATION — ALL 10 MODELS")
print("=" * 65)
print("Each model trains 15 times (LOSO folds)")
print("Estimated time: 15-30 minutes total")
print()

all_results = []

for model_name, model_class, model_params in models:

    print(f"Training {model_name}...", end=" ", flush=True)
    start = time.time()

    result = run_loso(
        model_class=model_class,
        model_params=model_params,
        X=X,
        y=y,
        subjects=subjects,
        feature_cols=feature_cols,
        model_name=model_name
    )

    elapsed = time.time() - start
    print(f"done ({elapsed:.0f}s)  "
          f"F1-macro={result['f1_macro']:.3f}  "
          f"F1-prestress={result['f1_prestress']:.3f}")

    all_results.append(result)

print()
print("=" * 65)
print("ALL MODELS TRAINED")
print("=" * 65)

RUNNING LOSO CROSS-VALIDATION — ALL 10 MODELS
Each model trains 15 times (LOSO folds)
Estimated time: 15-30 minutes total

Training Random Forest... done (20s)  F1-macro=0.585  F1-prestress=0.114
Training XGBoost... done (71s)  F1-macro=0.617  F1-prestress=0.286
Training SVM-RBF... done (1s)  F1-macro=0.573  F1-prestress=0.160
Training SVM-Linear... done (1s)  F1-macro=0.544  F1-prestress=0.205
Training KNN... done (9s)  F1-macro=0.544  F1-prestress=0.000
Training LDA... done (1s)  F1-macro=0.615  F1-prestress=0.202
Training AdaBoost... done (56s)  F1-macro=0.539  F1-prestress=0.121
Training MLP... done (14s)  F1-macro=0.583  F1-prestress=0.000
Training Logistic Regression... done (35s)  F1-macro=0.579  F1-prestress=0.201
Training Gradient Boosting... done (192s)  F1-macro=0.577  F1-prestress=0.145

ALL MODELS TRAINED


In [5]:
# ============================================================
# CELL 5 — RESULTS TABLE AND BEST MODEL SELECTION
#
# Prints full comparison table of all 10 models
# Selects best model based on:
#   Primary:   F1-Pre-stress (your novel contribution)
#   Secondary: F1-macro (overall performance)
#   Tertiary:  Ordinal MAE (error severity)
# ============================================================

print("=" * 85)
print("LOSO CROSS-VALIDATION RESULTS — ALL 10 MODELS")
print("=" * 85)
print(f"{'Model':22s} {'Accuracy':10s} {'F1-macro':10s} "
      f"{'F1-Normal':10s} {'F1-PreStr':10s} "
      f"{'F1-Stress':10s} {'OrdMAE':8s}")
print("-" * 85)

# Sort by F1-pre-stress then F1-macro
results_sorted = sorted(
    all_results,
    key=lambda x: (x['f1_prestress'], x['f1_macro']),
    reverse=True
)

for r in results_sorted:
    print(f"  {r['model_name']:20s} "
          f"{r['accuracy']:8.3f}   "
          f"{r['f1_macro']:8.3f}   "
          f"{r['f1_normal']:8.3f}   "
          f"{r['f1_prestress']:8.3f}   "
          f"{r['f1_stress']:8.3f}   "
          f"{r['ordinal_mae']:6.3f}")

print("-" * 85)

# Best model by F1-pre-stress
best = results_sorted[0]
print(f"\nBEST MODEL: {best['model_name']}")
print(f"  F1-macro:      {best['f1_macro']:.3f}")
print(f"  F1-Pre-stress: {best['f1_prestress']:.3f}")
print(f"  F1-Normal:     {best['f1_normal']:.3f}")
print(f"  F1-Stress:     {best['f1_stress']:.3f}")
print(f"  Ordinal MAE:   {best['ordinal_mae']:.3f}")
print(f"  Accuracy:      {best['accuracy']:.3f}")

# Targets check
print(f"\nTARGET CHECK:")
print(f"  F1-macro ≥ 0.80:      "
      f"{'PASS ✓' if best['f1_macro'] >= 0.80 else 'FAIL ✗'} "
      f"({best['f1_macro']:.3f})")
print(f"  F1-Pre-stress ≥ 0.75: "
      f"{'PASS ✓' if best['f1_prestress'] >= 0.75 else 'BELOW TARGET'} "
      f"({best['f1_prestress']:.3f})")
print(f"  Ordinal MAE < 0.50:   "
      f"{'PASS ✓' if best['ordinal_mae'] < 0.50 else 'FAIL ✗'} "
      f"({best['ordinal_mae']:.3f})")

# Detailed classification report for best model
print(f"\nDETAILED REPORT — {best['model_name']}:")
print(classification_report(
    best['all_true'],
    best['all_pred'],
    target_names=['Normal', 'Pre-stress', 'Stress'],
    digits=3
))

# Confusion matrix
print(f"CONFUSION MATRIX — {best['model_name']}:")
cm = confusion_matrix(best['all_true'], best['all_pred'])
print(f"                 Predicted")
print(f"                 Normal  Pre-str  Stress")
labels = ['Normal', 'Pre-str', 'Stress']
for i, row in enumerate(cm):
    print(f"  Actual {labels[i]:7s}: "
          f"{row[0]:6d}   {row[1]:6d}   {row[2]:6d}")

LOSO CROSS-VALIDATION RESULTS — ALL 10 MODELS
Model                  Accuracy   F1-macro   F1-Normal  F1-PreStr  F1-Stress  OrdMAE  
-------------------------------------------------------------------------------------
  XGBoost                 0.796      0.617      0.877      0.286      0.689    0.357
  SVM-Linear              0.728      0.544      0.878      0.205      0.550    0.403
  LDA                     0.838      0.615      0.935      0.202      0.708    0.243
  Logistic Regression     0.770      0.579      0.906      0.201      0.628    0.325
  SVM-RBF                 0.790      0.573      0.886      0.160      0.674    0.348
  Gradient Boosting       0.796      0.577      0.883      0.145      0.702    0.341
  AdaBoost                0.757      0.539      0.855      0.121      0.642    0.421
  Random Forest           0.830      0.585      0.894      0.114      0.748    0.305
  MLP                     0.883      0.583      0.936      0.000      0.813    0.200
  KNN           

In [6]:
# ============================================================
# CELL 6 — SAVE RESULTS AND BEST MODEL
# ============================================================

import joblib

print("=" * 65)
print("SAVING RESULTS AND BEST MODEL")
print("=" * 65)

# ── Save results table as CSV ─────────────────────────────────
results_rows = []
for r in all_results:
    results_rows.append({
        'model':        r['model_name'],
        'accuracy':     r['accuracy'],
        'f1_macro':     r['f1_macro'],
        'f1_normal':    r['f1_normal'],
        'f1_prestress': r['f1_prestress'],
        'f1_stress':    r['f1_stress'],
        'ordinal_mae':  r['ordinal_mae']
    })

results_df = pd.DataFrame(results_rows).sort_values(
    'f1_prestress', ascending=False
)

results_path = os.path.join(
    RESULTS_PATH, "objective1", "loso_results.csv"
)
results_df.to_csv(results_path, index=False)
print(f"Results table saved: {results_path}")

# ── Retrain best model on ALL data and save ───────────────────
# Why retrain on all data:
#   LOSO gave us evaluation metrics
#   For deployment we want model trained on all available data
#   More training data = better generalization
print(f"\nRetraining {best['model_name']} on all 15 subjects...")

# Get best model class and params
best_name = best['model_name']
best_class, best_params = None, None
for name, cls, params in models:
    if name == best_name:
        best_class  = cls
        best_params = params
        break

# Scale all data
scaler_final = StandardScaler()
X_scaled     = scaler_final.fit_transform(X)

# Train on all data
best_model_final = best_class(**best_params)
best_model_final.fit(X_scaled, y)

# Save model and scaler
models_path = os.path.join(
    COMPONENT_PATH, "saved_models"
)
os.makedirs(models_path, exist_ok=True)

model_save_path  = os.path.join(
    models_path, f"best_model_obj1.pkl"
)
scaler_save_path = os.path.join(
    models_path, f"scaler_obj1.pkl"
)
features_save_path = os.path.join(
    models_path, f"feature_cols_obj1.pkl"
)

joblib.dump(best_model_final, model_save_path)
joblib.dump(scaler_final,     scaler_save_path)
joblib.dump(feature_cols,     features_save_path)

print(f"Best model saved:  {model_save_path}")
print(f"Scaler saved:      {scaler_save_path}")
print(f"Feature cols saved:{features_save_path}")

# ── Print final summary ───────────────────────────────────────
print()
print("=" * 65)
print("OBJECTIVE 1 COMPLETE")
print("=" * 65)
print(f"Best model:        {best['model_name']}")
print(f"F1-macro:          {best['f1_macro']:.3f}")
print(f"F1-Pre-stress:     {best['f1_prestress']:.3f}")
print(f"Ordinal MAE:       {best['ordinal_mae']:.3f}")
print(f"Accuracy:          {best['accuracy']:.3f}")
print()
print("Files saved:")
print(f"  loso_results.csv")
print(f"  best_model_obj1.pkl")
print(f"  scaler_obj1.pkl")
print(f"  feature_cols_obj1.pkl")
print()
print("Next step: Objective 2 — Personalized adaptive model")

SAVING RESULTS AND BEST MODEL
Results table saved: C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals\results\objective1\loso_results.csv

Retraining XGBoost on all 15 subjects...
Best model saved:  C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals\saved_models\best_model_obj1.pkl
Scaler saved:      C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals\saved_models\scaler_obj1.pkl
Feature cols saved:C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals\saved_models\feature_cols_obj1.pkl

OBJECTIVE 1 COMPLETE
Best model:        XGBoost
F1-macro:          0.617
F1-Pre-stress:     0.286
Ordinal MAE:       0.357
Accuracy:          0.796

Files saved:
  loso_results.csv
  best_model_obj1.pkl
  scaler_obj1.pkl
  feature_cols_obj1.pkl

Next step: Objective 2 — Personalized adaptive model


In [7]:
# ============================================================
# CELL 7 — IMPROVED XGBOOST WITH SAMPLE WEIGHTS
# ============================================================

from sklearn.utils.class_weight import compute_sample_weight

print("=" * 65)
print("IMPROVED XGBOOST WITH SAMPLE WEIGHTS")
print("=" * 65)

def run_loso_weighted(X, y, subjects, feature_cols):
    """
    LOSO with XGBoost using sample weights.
    Sample weights give pre-stress windows 18x more importance.
    """

    unique_subjects = sorted(np.unique(subjects))
    all_true        = []
    all_pred        = []

    for sid in unique_subjects:

        test_mask  = subjects == sid
        train_mask = ~test_mask

        X_train = X[train_mask]
        y_train = y[train_mask]
        X_test  = X[test_mask]
        y_test  = y[test_mask]

        # Scale inside fold
        scaler  = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

        # Compute sample weights — pre-stress gets highest weight
        sample_weights = compute_sample_weight(
            'balanced', y_train
        )

        # XGBoost with sample weights
        model = xgb.XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            use_label_encoder=False,
            eval_metric='mlogloss',
            random_state=42,
            n_jobs=-1,
            verbosity=0
        )
        model.fit(
            X_train, y_train,
            sample_weight=sample_weights
        )

        y_pred = model.predict(X_test)
        all_true.extend(y_test.tolist())
        all_pred.extend(y_pred.tolist())

    all_true = np.array(all_true)
    all_pred = np.array(all_pred)

    f1_mac = f1_score(all_true, all_pred,
                      average='macro', zero_division=0)
    f1_pre = f1_score(all_true, all_pred,
                      labels=[1], average='macro',
                      zero_division=0)
    f1_nor = f1_score(all_true, all_pred,
                      labels=[0], average='macro',
                      zero_division=0)
    f1_str = f1_score(all_true, all_pred,
                      labels=[2], average='macro',
                      zero_division=0)
    mae    = np.mean(np.abs(all_true - all_pred))
    acc    = np.mean(all_true == all_pred)

    print(f"\nXGBoost with sample weights:")
    print(f"  F1-macro:      {f1_mac:.3f}")
    print(f"  F1-Pre-stress: {f1_pre:.3f}")
    print(f"  F1-Normal:     {f1_nor:.3f}")
    print(f"  F1-Stress:     {f1_str:.3f}")
    print(f"  Ordinal MAE:   {mae:.3f}")
    print(f"  Accuracy:      {acc:.3f}")

    print(f"\nDetailed report:")
    print(classification_report(
        all_true, all_pred,
        target_names=['Normal','Pre-stress','Stress'],
        digits=3
    ))

    print(f"Confusion matrix:")
    cm = confusion_matrix(all_true, all_pred)
    print(f"                 Normal  Pre-str  Stress")
    for i, row in enumerate(cm):
        lbl = ['Normal','Pre-str','Stress'][i]
        print(f"  Actual {lbl:7s}: "
              f"{row[0]:6d}   {row[1]:6d}   {row[2]:6d}")

    return all_true, all_pred, f1_mac, f1_pre

print("Running XGBoost with sample weights...")
t, p, f1m, f1p = run_loso_weighted(
    X, y, subjects, feature_cols
)

IMPROVED XGBOOST WITH SAMPLE WEIGHTS
Running XGBoost with sample weights...

XGBoost with sample weights:
  F1-macro:      0.617
  F1-Pre-stress: 0.235
  F1-Normal:     0.909
  F1-Stress:     0.706
  Ordinal MAE:   0.296
  Accuracy:      0.815

Detailed report:
              precision    recall  f1-score   support

      Normal      0.927     0.891     0.909       587
  Pre-stress      0.179     0.345     0.235        29
      Stress      0.711     0.700     0.706       267

    accuracy                          0.815       883
   macro avg      0.606     0.645     0.617       883
weighted avg      0.837     0.815     0.825       883

Confusion matrix:
                 Normal  Pre-str  Stress
  Actual Normal :    523        4       60
  Actual Pre-str:      3       10       16
  Actual Stress :     38       42      187


In [8]:
# ============================================================
# CELL 8 — THRESHOLD TUNING FOR PRE-STRESS
#
# Instead of using argmax prediction directly
# Lower the threshold needed to predict pre-stress
# This increases pre-stress recall at cost of precision
# ============================================================

print("=" * 65)
print("THRESHOLD TUNING FOR PRE-STRESS")
print("=" * 65)

def run_loso_threshold(X, y, subjects,
                       prestress_threshold=0.15):
    """
    LOSO with probability threshold for pre-stress.
    If P(pre-stress) > threshold → predict pre-stress
    Default argmax would need P > 0.5
    Lower threshold → catches more pre-stress cases
    """

    unique_subjects = sorted(np.unique(subjects))
    all_true = []
    all_pred = []

    for sid in unique_subjects:

        test_mask  = subjects == sid
        train_mask = ~test_mask

        X_train = X[train_mask]
        y_train = y[train_mask]
        X_test  = X[test_mask]
        y_test  = y[test_mask]

        scaler  = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

        sample_weights = compute_sample_weight(
            'balanced', y_train
        )

        model = xgb.XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            use_label_encoder=False,
            eval_metric='mlogloss',
            random_state=42,
            n_jobs=-1,
            verbosity=0
        )
        model.fit(
            X_train, y_train,
            sample_weight=sample_weights
        )

        # Get probabilities instead of hard prediction
        y_proba = model.predict_proba(X_test)
        # y_proba shape: (n_samples, 3)
        # columns: [P(Normal), P(Pre-stress), P(Stress)]

        # Apply threshold: if P(pre-stress) > threshold
        # predict pre-stress regardless of other classes
        y_pred = np.argmax(y_proba, axis=1)
        prestress_mask = y_proba[:, 1] > prestress_threshold
        y_pred[prestress_mask] = 1

        all_true.extend(y_test.tolist())
        all_pred.extend(y_pred.tolist())

    all_true = np.array(all_true)
    all_pred = np.array(all_pred)

    f1_mac = f1_score(all_true, all_pred,
                      average='macro', zero_division=0)
    f1_pre = f1_score(all_true, all_pred,
                      labels=[1], average='macro',
                      zero_division=0)
    mae    = np.mean(np.abs(all_true - all_pred))

    return f1_mac, f1_pre, mae, all_true, all_pred


# Test different thresholds
print(f"{'Threshold':12s} {'F1-macro':10s} "
      f"{'F1-PreStr':10s} {'OrdMAE':8s}")
print("-" * 45)

best_threshold = 0.15
best_f1_pre    = 0
best_results   = None

for threshold in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    f1m, f1p, mae, t, p = run_loso_threshold(
        X, y, subjects, threshold
    )
    marker = " ← best" if f1p > best_f1_pre else ""
    print(f"  {threshold:.2f}         "
          f"{f1m:.3f}      "
          f"{f1p:.3f}      "
          f"{mae:.3f}{marker}")

    if f1p > best_f1_pre:
        best_f1_pre    = f1p
        best_threshold = threshold
        best_results   = (t, p, f1m, f1p, mae)

print(f"\nBest threshold: {best_threshold}")
t, p, f1m, f1p, mae = best_results
print(f"  F1-macro:      {f1m:.3f}")
print(f"  F1-Pre-stress: {f1p:.3f}")
print(f"  Ordinal MAE:   {mae:.3f}")

print(f"\nDetailed report at threshold={best_threshold}:")
print(classification_report(
    t, p,
    target_names=['Normal','Pre-stress','Stress'],
    digits=3
))

THRESHOLD TUNING FOR PRE-STRESS
Threshold    F1-macro   F1-PreStr  OrdMAE  
---------------------------------------------
  0.05         0.567      0.164      0.323 ← best
  0.10         0.577      0.169      0.315 ← best
  0.15         0.587      0.189      0.314 ← best
  0.20         0.600      0.211      0.304 ← best
  0.25         0.606      0.224      0.302 ← best
  0.30         0.611      0.233      0.298 ← best

Best threshold: 0.3
  F1-macro:      0.611
  F1-Pre-stress: 0.233
  Ordinal MAE:   0.298

Detailed report at threshold=0.3:
              precision    recall  f1-score   support

      Normal      0.929     0.889     0.909       587
  Pre-stress      0.162     0.414     0.233        29
      Stress      0.721     0.667     0.693       267

    accuracy                          0.806       883
   macro avg      0.604     0.657     0.611       883
weighted avg      0.841     0.806     0.821       883



In [9]:
# ============================================================
# CELL 9 — VOTING ENSEMBLE (TOP 3 MODELS)
#
# Combine XGBoost + Random Forest + LDA
# Majority vote across 3 models
# Often better than any single model
# ============================================================

from sklearn.ensemble import VotingClassifier

print("=" * 65)
print("VOTING ENSEMBLE — XGBoost + RF + LDA")
print("=" * 65)

def run_loso_ensemble(X, y, subjects):
    """
    LOSO with soft voting ensemble.
    XGBoost + Random Forest + LDA combined.
    """

    unique_subjects = sorted(np.unique(subjects))
    all_true = []
    all_pred = []

    for sid in unique_subjects:

        test_mask  = subjects == sid
        train_mask = ~test_mask

        X_train = X[train_mask]
        y_train = y[train_mask]
        X_test  = X[test_mask]
        y_test  = y[test_mask]

        scaler  = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

        sample_weights = compute_sample_weight(
            'balanced', y_train
        )

        # Model 1: XGBoost
        m1 = xgb.XGBClassifier(
            n_estimators=300, max_depth=6,
            learning_rate=0.1,
            use_label_encoder=False,
            eval_metric='mlogloss',
            random_state=42, n_jobs=-1, verbosity=0
        )

        # Model 2: Random Forest
        m2 = RandomForestClassifier(
            n_estimators=300,
            class_weight='balanced',
            random_state=42, n_jobs=-1
        )

        # Model 3: LDA
        m3 = LinearDiscriminantAnalysis(solver='svd')

        # Train all three
        m1.fit(X_train, y_train,
               sample_weight=sample_weights)
        m2.fit(X_train, y_train)
        m3.fit(X_train, y_train)

        # Get probabilities from each
        p1 = m1.predict_proba(X_test)
        p2 = m2.predict_proba(X_test)
        try:
            p3 = m3.predict_proba(X_test)
            # Average probabilities (soft voting)
            p_avg  = (p1 + p2 + p3) / 3
        except Exception:
            p_avg  = (p1 + p2) / 2

        y_pred = np.argmax(p_avg, axis=1)

        all_true.extend(y_test.tolist())
        all_pred.extend(y_pred.tolist())

    all_true = np.array(all_true)
    all_pred = np.array(all_pred)

    f1_mac = f1_score(all_true, all_pred,
                      average='macro', zero_division=0)
    f1_pre = f1_score(all_true, all_pred,
                      labels=[1], average='macro',
                      zero_division=0)
    f1_nor = f1_score(all_true, all_pred,
                      labels=[0], average='macro',
                      zero_division=0)
    f1_str = f1_score(all_true, all_pred,
                      labels=[2], average='macro',
                      zero_division=0)
    mae    = np.mean(np.abs(all_true - all_pred))
    acc    = np.mean(all_true == all_pred)

    print(f"Voting Ensemble (XGB + RF + LDA):")
    print(f"  F1-macro:      {f1_mac:.3f}")
    print(f"  F1-Pre-stress: {f1_pre:.3f}")
    print(f"  F1-Normal:     {f1_nor:.3f}")
    print(f"  F1-Stress:     {f1_str:.3f}")
    print(f"  Ordinal MAE:   {mae:.3f}")
    print(f"  Accuracy:      {acc:.3f}")

    print(f"\nDetailed report:")
    print(classification_report(
        all_true, all_pred,
        target_names=['Normal','Pre-stress','Stress'],
        digits=3
    ))

    print(f"Confusion matrix:")
    cm = confusion_matrix(all_true, all_pred)
    print(f"                 Normal  Pre-str  Stress")
    for i, row in enumerate(cm):
        lbl = ['Normal','Pre-str','Stress'][i]
        print(f"  Actual {lbl:7s}: "
              f"{row[0]:6d}   {row[1]:6d}   {row[2]:6d}")

    return all_true, all_pred, f1_mac, f1_pre


print("Running ensemble (takes ~5 minutes)...")
t, p, f1m, f1p = run_loso_ensemble(X, y, subjects)

print(f"\nCOMPARISON SUMMARY:")
print(f"{'Model':25s} {'F1-macro':10s} "
      f"{'F1-PreStr':10s} {'OrdMAE':8s}")
print("-" * 55)
print(f"  {'XGBoost (original)':23s} "
      f"0.617       0.286       0.357")
print(f"  {'XGBoost + weights':23s} "
      f"0.617       0.235       0.296")
print(f"  {'Threshold tuned':23s} "
      f"0.611       0.233       0.298")
print(f"  {'Voting Ensemble':23s} "
      f"{f1m:.3f}       {f1p:.3f}       "
      f"{np.mean(np.abs(t-p)):.3f}")

VOTING ENSEMBLE — XGBoost + RF + LDA
Running ensemble (takes ~5 minutes)...
Voting Ensemble (XGB + RF + LDA):
  F1-macro:      0.645
  F1-Pre-stress: 0.246
  F1-Normal:     0.927
  F1-Stress:     0.762
  Ordinal MAE:   0.239
  Accuracy:      0.856

Detailed report:
              precision    recall  f1-score   support

      Normal      0.915     0.939     0.927       587
  Pre-stress      0.250     0.241     0.246        29
      Stress      0.783     0.742     0.762       267

    accuracy                          0.856       883
   macro avg      0.649     0.641     0.645       883
weighted avg      0.853     0.856     0.854       883

Confusion matrix:
                 Normal  Pre-str  Stress
  Actual Normal :    551        0       36
  Actual Pre-str:      3        7       19
  Actual Stress :     48       21      198

COMPARISON SUMMARY:
Model                     F1-macro   F1-PreStr  OrdMAE  
-------------------------------------------------------
  XGBoost (original)      0.617